In [2]:
import os, shutil, sys

REPO_NAME = "standard-kernel-takehome"   # the repo

# nuke any partial clone from before
if os.path.exists(REPO_NAME):
    shutil.rmtree(REPO_NAME)

os.chdir("/content")

!git clone https://github.com/andrewscoding2018/standard-kernel-takehome

assert os.path.exists(REPO_NAME), "clone failed"
os.chdir(REPO_NAME)
print("cwd:", os.getcwd())
print("contents:", os.listdir("."))
!pip install -e . --quiet

Cloning into 'standard-kernel-takehome'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 211 (delta 150), reused 122 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 337.72 KiB | 15.35 MiB/s, done.
Resolving deltas: 100% (150/150), done.
cwd: /content/standard-kernel-takehome
contents: ['tests', 'notebooks', '.gitignore', '.git', 'eval', 'uv.lock', 'benchmarks', 'README.md', 'main.py', '.DS_Store', '.python-version', 'main.ipynb', 'fp4_kernel', 'pyproject.toml']
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fp4_kernel (pyproject.toml) ... done


In [2]:
!pytest tests/test_quant.py -v

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
# default code from transformers website

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=120)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [1]:
import os

os.environ["TRITON_INTERPRET"] = "1"

import torch, triton, triton.language as tl
import numpy as np

In [3]:
from fp4_kernel.formats import FP4_E2M1
from fp4_kernel.quant import quantize, dequantize

block_size = 8
M, N, K = 2, 4, block_size

x = torch.arange(M * K, dtype = torch.float32).reshape(M, K) * 0.1
W = torch.randn(N, K)

qt = quantize(W, format = FP4_E2M1, block_size = block_size, scale_mode = "absmax")
print("packed", qt.packed.shape, qt.packed.dtype)
print("scales", qt.scales.shape)
print("LUT", qt.fmt.code_to_value)

packed torch.Size([4, 1, 4]) torch.uint8
scales torch.Size([4, 1])
LUT [ 0.   0.5  1.   1.5  2.   3.   4.   6.   0.  -0.5 -1.  -1.5 -2.  -3.
 -4.  -6. ]


In [7]:
@triton.jit
def fp4_matmul_kernel_dbg(x_ptr, packed_ptr, scales_ptr, out_ptr,
                          M, N, K, n_blocks,
                          stride_xm, stride_xk, stride_wn, stride_wk,
                          stride_sn, stride_sk, code_to_value_ptr,
                          BLOCK_SIZE: tl.constexpr, HALF: tl.constexpr, BM: tl.constexpr, BN: tl.constexpr):
    pid_m, pid_n = tl.program_id(0), tl.program_id(1)
    offs_m = pid_m * BM + tl.arange(0, BM)
    offs_n = pid_n * BN + tl.arange(0, BN)
    offs_j = tl.arange(0, HALF)
    acc = tl.zeros((BM, BN), dtype=tl.float32)

    for kb in range(n_blocks):
        w_byte_ptrs = packed_ptr + offs_n[None, :]*stride_wn + (kb*HALF + offs_j[:, None])*stride_wk
        b = tl.load(w_byte_ptrs, mask=offs_n[None, :] < N, other=0)
        low  = (b & 0xF).to(tl.int32)
        high = ((b >> 4) & 0xF).to(tl.int32)
        print("kb", kb, "raw bytes\n", b)             # <-- packed nibbles
        print("low codes\n", low, "\nhigh codes\n", high)

        w_low  = tl.load(code_to_value_ptr + low)
        w_high = tl.load(code_to_value_ptr + high)
        scale  = tl.load(scales_ptr + offs_n*stride_sn + kb*stride_sk, mask=offs_n < N, other=0.0)
        w_low  = (w_low  * scale[None, :]).to(tl.float32)
        w_high = (w_high * scale[None, :]).to(tl.float32)
        print("decoded*scale w_low\n", w_low)         # <-- should equal dequant(W) cols

        k_even = kb*BLOCK_SIZE + 2*offs_j
        k_odd  = k_even + 1
        x_even = tl.load(x_ptr + offs_m[:, None]*stride_xm + k_even[None, :]*stride_xk,
                         mask=(offs_m[:, None] < M) & (k_even[None, :] < K), other=0.0)
        x_odd  = tl.load(x_ptr + offs_m[:, None]*stride_xm + k_odd[None, :]*stride_xk,
                         mask=(offs_m[:, None] < M) & (k_odd[None, :] < K), other=0.0)
        print("x_even\n", x_even, "\nx_odd\n", x_odd)  # <-- the interleave split

        acc += tl.dot(x_even.to(tl.float32), w_low)
        acc += tl.dot(x_odd.to(tl.float32),  w_high)
        print("acc after kb", kb, "\n", acc)

    out_ptrs = out_ptr + offs_m[:, None]*N + offs_n[None, :]
    tl.store(out_ptrs, acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

In [8]:
packed2d = qt.packed.reshape(N, -1).contiguous().to(torch.uint8)
scales   = qt.scales.reshape(N, -1).contiguous().float()
lut      = torch.tensor(qt.fmt.code_to_value, dtype=torch.float32)
out      = torch.empty(M, N, dtype=torch.float32)

fp4_matmul_kernel_dbg[(1, 1)](                      # grid = one program
    x, packed2d, scales, out, M, N, K, scales.shape[-1],
    x.stride(0), x.stride(1), packed2d.stride(0), packed2d.stride(1),
    scales.stride(0), scales.stride(1), lut,
    BLOCK_SIZE=block_size, HALF=block_size // 2, BM=M, BN=N)

kb 0 raw bytes
 [[ 35 213 155  68]
 [ 28  87 255  38]
 [ 15  84 116 158]
 [ 68  31  62 116]]
low codes
 [[ 3  5 11  4]
 [12  7 15  6]
 [15  4  4 14]
 [ 4 15 14  4]] 
high codes
 [[ 2 13  9  4]
 [ 1  5 15  2]
 [ 0  5  7  9]
 [ 4  1  3  7]]
decoded*scale w_low
 [[ 0.47377056  0.63922715 -0.33550805  0.5649753 ]
 [-0.6316941   1.2784543  -1.3420322   1.1299506 ]
 [-1.8950822   0.42615142  0.44734406 -1.1299506 ]
 [ 0.6316941  -1.2784543  -0.8946881   0.5649753 ]]
x_even
 [[0.  0.2 0.4 0.6]
 [0.8 1.  1.2 1.4]] 
x_odd
 [[0.1        0.3        0.5        0.7       ]
 [0.90000004 1.1        1.3000001  1.5       ]]
acc after kb 0 
 [[ 0.01579231  0.18111438 -0.13420328  1.3700651 ]
 [-0.23688531  1.6300294  -1.6551728   4.1949415 ]]
